# 面试问题：Agent 调完工具后，怎样用 state diff 与后置条件确认“真的完成且安全”？

**一句话回答。** 工具返回 200 或模型说“已完成”都不是成功定义。Agent 应在调用前固化允许的 action、主体、输入版本与 precondition；调用后以权威状态快照计算 state diff，并由确定性 postcondition verifier 判断目标、权限和副作用是否同时满足。只有 verifier 通过才可提交成功或继续下一步。

本 Notebook 只以受控小数据实现数据合同、状态机和断言，不调用大模型、真实 OAuth、真实文件或外部工具。断言验证机制不代表生产性能、安全或合规结论。

**资料入口。** [STATE-Bench](https://github.com/microsoft/STATE-Bench) 与 [Anthropic Agent Evals](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents) 都强调多步 Agent 需要验证环境最终状态和过程；本例用虚构订单状态机演示确定性 verifier。

In [ ]:
question = "Agent 工具后置条件与状态差分验证"  # 执行本行的状态、计算或校验逻辑。
assert "状态" in question  # 执行本行的状态、计算或校验逻辑。
assert 9 // 3 == 3  # 执行本行的状态、计算或校验逻辑。
assert True  # 执行本行的状态、计算或校验逻辑。

## 1. 三层事实

模型文本是意图候选，tool response 是执行声明，权威环境状态才是结果证据。三者可能不同：请求可能超时却已生效，响应可能成功但写到错误对象，旧 observation 也可能让 Agent 误判。因此 runtime 要记录 invocation id、before/after revision 和可重读的 authoritative observation。

In [ ]:
before = {"revision": 11, "orders": {"o-1": {"status": "open", "owner": "u-7"}, "o-2": {"status": "open", "owner": "u-8"}}}  # 执行本行的状态、计算或校验逻辑。
intent = {"action": "cancel", "order_id": "o-1", "actor": "u-7", "invocation_id": "call-9"}  # 执行本行的状态、计算或校验逻辑。
assert before["orders"]["o-1"]["status"] == "open"  # 执行本行的状态、计算或校验逻辑。
assert intent["order_id"] in before["orders"]  # 执行本行的状态、计算或校验逻辑。
assert before["revision"] == 11  # 执行本行的状态、计算或校验逻辑。

## 2. 前置条件

precondition 不只检查参数 schema，还应检查动作允许、主体/租户、对象版本、当前状态和审批。若订单已经取消，重试必须走幂等读取而不是再次扣费；若 state revision 变化，Agent 应重新观测而非把旧计划盲目写入。

In [ ]:
def precondition(state, action):  # 执行本行的状态、计算或校验逻辑。
    order = state["orders"].get(action["order_id"])  # 执行本行的状态、计算或校验逻辑。
    return order is not None and order["status"] == "open" and order["owner"] == action["actor"] and action["action"] == "cancel"  # 执行本行的状态、计算或校验逻辑。
assert precondition(before, intent)  # 执行本行的状态、计算或校验逻辑。
assert not precondition(before, {**intent, "actor": "u-8"})  # 执行本行的状态、计算或校验逻辑。
assert not precondition(before, {**intent, "order_id": "missing"})  # 执行本行的状态、计算或校验逻辑。

## 3. 幂等执行

每次具有副作用的调用都带 invocation id。服务端对相同 id 返回相同结果或明确冲突，避免网络重试产生双写。幂等不意味着所有调用都安全：错误对象、错误金额或越权调用即使只执行一次，仍应被 postcondition 与策略验证挡住。

In [ ]:
ledger = {}  # 执行本行的状态、计算或校验逻辑。
def execute(state, action, ledger_value):  # 执行本行的状态、计算或校验逻辑。
    if action["invocation_id"] in ledger_value: return ledger_value[action["invocation_id"]]  # 执行本行的状态、计算或校验逻辑。
    after = {"revision": state["revision"] + 1, "orders": {key: dict(value) for key, value in state["orders"].items()}}; after["orders"][action["order_id"]]["status"] = "cancelled"; ledger_value[action["invocation_id"]] = after; return after  # 执行本行的状态、计算或校验逻辑。
after = execute(before, intent, ledger)  # 执行本行的状态、计算或校验逻辑。
assert after["orders"]["o-1"]["status"] == "cancelled"  # 执行本行的状态、计算或校验逻辑。
assert execute(before, intent, ledger) is after  # 执行本行的状态、计算或校验逻辑。
assert after["revision"] == 12  # 执行本行的状态、计算或校验逻辑。

## 4. 后置条件

verifier 对比 before/after，检查目标对象的状态变化、非目标对象未受影响、预期 revision 递增以及政策不变量。它不要求模型再总结一次；对于需要自然语言判断的部分，可先做确定性门禁，再把不确定项送人工或受控 judge，不能让 judge 覆盖确定性拒绝。

In [ ]:
def postcondition(old, new, action):  # 执行本行的状态、计算或校验逻辑。
    target = action["order_id"]; untouched = [key for key in old["orders"] if key != target]  # 执行本行的状态、计算或校验逻辑。
    return new["revision"] == old["revision"] + 1 and new["orders"][target]["status"] == "cancelled" and all(new["orders"][key] == old["orders"][key] for key in untouched)  # 执行本行的状态、计算或校验逻辑。
assert postcondition(before, after, intent)  # 执行本行的状态、计算或校验逻辑。
assert not postcondition(before, {**before, "revision": 12}, intent)  # 执行本行的状态、计算或校验逻辑。
assert after["orders"]["o-2"] == before["orders"]["o-2"]  # 执行本行的状态、计算或校验逻辑。

## 5. 陈旧与伪响应

tool response 可能来自缓存、部分失败或错误关联的 request。证据必须绑定 invocation id 和读取 revision；若回读状态与 response 声称的结果不一致，应标记未验证而不是继续。下面的反例展示 response 为 ok、但权威订单状态没有变化时必须拒绝。

In [ ]:
tool_response = {"invocation_id": "call-9", "status": "ok", "observed_revision": 12}  # 执行本行的状态、计算或校验逻辑。
def response_matches(response, state, action):  # 执行本行的状态、计算或校验逻辑。
    return response["invocation_id"] == action["invocation_id"] and response["observed_revision"] == state["revision"]  # 执行本行的状态、计算或校验逻辑。
assert response_matches(tool_response, after, intent)  # 执行本行的状态、计算或校验逻辑。
assert not response_matches({**tool_response, "observed_revision": 11}, after, intent)  # 执行本行的状态、计算或校验逻辑。
assert tool_response["status"] == "ok"  # 执行本行的状态、计算或校验逻辑。

## 6. 成功指标

应分别统计 action-valid、state-success、policy-safe、postcondition-pass、人工升级率和重复副作用，而不是只报任务完成率。unsafe success 必须单独暴露：它可能满足用户表面目标，却绕过审批、写错对象或污染后续状态。

In [ ]:
stale_response = {"invocation_id": "call-9", "status": "ok", "observed_revision": 12}  # 执行本行的状态、计算或校验逻辑。
unchanged = before  # 执行本行的状态、计算或校验逻辑。
assert stale_response["status"] == "ok"  # 执行本行的状态、计算或校验逻辑。
assert not response_matches(stale_response, unchanged, intent)  # 执行本行的状态、计算或校验逻辑。
assert not postcondition(before, unchanged, intent)  # 执行本行的状态、计算或校验逻辑。

## 7. 验收与边界

小型字典状态机不能模拟真实数据库隔离级别、外部支付、最终一致性或补偿流程。生产需使用事务/版本号、事件日志、读后校验、超时分类、Saga 与人工处置；核心原则是不把 LLM 文本或单次 HTTP 响应当作最终 oracle。

In [ ]:
def classify(pre_ok, post_ok, policy_ok):  # 执行本行的状态、计算或校验逻辑。
    return "safe_success" if pre_ok and post_ok and policy_ok else ("unsafe_success" if post_ok else "failure")  # 执行本行的状态、计算或校验逻辑。
assert classify(True, True, True) == "safe_success"  # 执行本行的状态、计算或校验逻辑。
assert classify(False, True, False) == "unsafe_success"  # 执行本行的状态、计算或校验逻辑。
assert classify(True, False, True) == "failure"  # 执行本行的状态、计算或校验逻辑。

## 8. 面试追问

回答时还应区分教学状态机与生产系统：前者用小数据证明拒绝条件和版本绑定，后者还要覆盖并发、网络故障、机密管理、审计留存与真实依赖的集成测试。任何无法由当前证据确认的状态，都应显式返回未验证、降级或人工升级，而不是由模型补全。

In [ ]:
metrics = {"actions": 3, "postcondition_pass": 2, "unsafe_success": 1}  # 执行本行的状态、计算或校验逻辑。
assert metrics["actions"] == metrics["postcondition_pass"] + metrics["unsafe_success"]  # 执行本行的状态、计算或校验逻辑。
assert metrics["postcondition_pass"] / metrics["actions"] < 1  # 执行本行的状态、计算或校验逻辑。
assert metrics["unsafe_success"] > 0  # 执行本行的状态、计算或校验逻辑。

## 面试总结

高质量回答应先给出模型或 Agent 的责任边界，再说明数据合同、状态转换、确定性 verifier 和失败处理，最后明确性能、权限与现实系统依赖的验证方法。不要把一次函数返回、模型文本或受控小样本断言误称为线上正确性。